In [3]:
import pandas as pd
df4= pd.read_csv(r'C:\Users\daniealv\Downloads\prueba_op_maestra_cuotas_pagos_mes_hist_enmascarado_completa.csv')

In [4]:
tipos=df4.dtypes
describe=df4.describe()

In [5]:
missing_values = df4.isnull().sum()
print(missing_values)

nit_enmascarado              0
num_oblig_enmascarado        0
fecha_corte                  0
producto                     0
aplicativo                   0
segmento                     0
valor_cuota_mes              0
pago_total                   0
fecha_pago_minima            0
fecha_pago_maxima            0
porc_pago                77376
marca_pago                   0
ajustes_banco                0
dtype: int64


In [12]:

df= pd.read_csv(r'C:\Users\daniealv\Downloads\prueba_op_base_pivot_var_rpta_alt_enmascarado_trtest.csv')
extracted_df = df[['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','var_rpta_alt','fecha_var_rpta_alt']]
merged_df = pd.merge(extracted_df, df4, how='left', on=['nit_enmascarado', 'num_oblig_enmascarado'])

c:\Users\daniealv\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (28,37,38) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
# Convertir las columnas 'fecha_corte' y 'fecha_var_rpta_alt' en tipo datetime
merged_df['fecha_corte'] = pd.to_datetime(merged_df['fecha_corte'], format='%Y%m%d')
merged_df['fecha_var_rpta_alt'] = pd.to_datetime(merged_df['fecha_var_rpta_alt'], format='%Y%m')
merged_df['fecha_corte'] = merged_df['fecha_corte'].apply(lambda x: x.replace(day=1))
# Agrupar por las columnas especificadas y contar las ocurrencias de cada valor en 'marca_pago'
marca_pago_counts = merged_df.groupby(['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado', 'fecha_var_rpta_alt', 'marca_pago']).size().unstack(fill_value=0).reset_index()

# Renombrar las columnas para que sean más descriptivas
marca_pago_counts.columns = ['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado', 'fecha_var_rpta_alt'] + [f'count_{col}' for col in marca_pago_counts.columns[4:]]


In [14]:
merged_df = merged_df[merged_df['fecha_var_rpta_alt'] > merged_df['fecha_corte']]
merged_df = merged_df.drop(columns=['fecha_pago_minima', 'fecha_pago_maxima'])
# Ordenar el DataFrame por 'num_oblig_enmascarado', 'fecha_var_rpta_alt' y 'fecha_corte'
merged_df = merged_df.sort_values(by=['num_oblig_enmascarado','num_oblig_orig_enmascarado', 'fecha_var_rpta_alt', 'fecha_corte'])

# Tomar los últimos seis pagos para cada obligación
merged_df = merged_df.groupby(['num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt']).tail(6)
merged_df['numerador'] = merged_df.groupby(['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt']).cumcount() + 1
merged_df['ratio_pago_cuota'] = merged_df['pago_total'] / merged_df['valor_cuota_mes']
pivot_df = merged_df.pivot_table(index=['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'], columns='numerador', values='ratio_pago_cuota').reset_index()

# Renombrar las columnas para que sean más descriptivas
pivot_df.columns = ['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'] + [f'ratio_pago_cuota_{i}' for i in range(1, 7)]
extracted_df['fecha_var_rpta_alt'] = pd.to_datetime(extracted_df['fecha_var_rpta_alt'], format='%Y%m')
# Realizar un merge para encontrar los registros en extracted_df que no están en pivot_df
merged_check = pd.merge(extracted_df, pivot_df, how='left', on=['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','fecha_var_rpta_alt'])


C:\Users\daniealv\AppData\Local\Temp/ipykernel_32956/429409883.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  extracted_df['fecha_var_rpta_alt'] = pd.to_datetime(extracted_df['fecha_var_rpta_alt'], format='%Y%m')


In [15]:
merged_check = pd.merge(merged_check, marca_pago_counts, on=['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado', 'fecha_var_rpta_alt'], how='left')
ratio_columns = [col for col in merged_check.columns if col.startswith('ratio_pago_')]
merged_check[ratio_columns] = merged_check[ratio_columns].fillna(0)
# Obtener el aplicativo y segmento cuando el numerador sea el mayor por cada nit, num_oblig_enmascarado y num_oblig_orig_enmascarado
max_numerador_df = merged_df.loc[merged_df.groupby(['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','fecha_var_rpta_alt'])['numerador'].idxmax(), ['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado', 'aplicativo', 'segmento','fecha_var_rpta_alt']]


In [16]:
merged_check2 = pd.merge(merged_check, max_numerador_df, on=['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','fecha_var_rpta_alt'], how='left')

In [17]:
# Seleccionar las columnas de interés
columns_of_interest = ['ratio_pago_cuota_1', 'ratio_pago_cuota_2', 'ratio_pago_cuota_3', 'ratio_pago_cuota_4', 'ratio_pago_cuota_5', 'ratio_pago_cuota_6', 'var_rpta_alt']

# Filtrar el DataFrame para incluir solo las columnas de interés
df_corr = merged_check2[columns_of_interest]

# Calcular la correlación
correlation_matrix = df_corr.corr()

# Mostrar la matriz de correlación
print(correlation_matrix)

                    ratio_pago_cuota_1  ratio_pago_cuota_2  \
ratio_pago_cuota_1            1.000000            0.003431   
ratio_pago_cuota_2            0.003431            1.000000   
ratio_pago_cuota_3            0.000225            0.039865   
ratio_pago_cuota_4            0.000078            0.000093   
ratio_pago_cuota_5            0.000152            0.000066   
ratio_pago_cuota_6           -0.000014            0.000026   
var_rpta_alt                  0.000687           -0.000819   

                    ratio_pago_cuota_3  ratio_pago_cuota_4  \
ratio_pago_cuota_1            0.000225            0.000078   
ratio_pago_cuota_2            0.039865            0.000093   
ratio_pago_cuota_3            1.000000            0.000024   
ratio_pago_cuota_4            0.000024            1.000000   
ratio_pago_cuota_5            0.000047            0.000009   
ratio_pago_cuota_6           -0.000005           -0.000007   
var_rpta_alt                 -0.001065           -0.001603   

      

In [18]:
# Seleccionar las columnas de interés
columns_of_interest_corr = ['var_rpta_alt', 'count_FACTURACION_MES_SGTE', 'count_PAGO_MENOS', 'count_PAGO_MAS']

# Filtrar el DataFrame para incluir solo las columnas de interés
df_corr_facturacion = merged_check2[columns_of_interest_corr]

# Calcular la correlación
correlation_matrix_facturacion = df_corr_facturacion.corr()

# Mostrar la matriz de correlación
print(correlation_matrix_facturacion)

                            var_rpta_alt  count_FACTURACION_MES_SGTE  \
var_rpta_alt                    1.000000                    0.098396   
count_FACTURACION_MES_SGTE      0.098396                    1.000000   
count_PAGO_MENOS               -0.053305                   -0.217946   
count_PAGO_MAS                  0.143257                   -0.002546   

                            count_PAGO_MENOS  count_PAGO_MAS  
var_rpta_alt                       -0.053305        0.143257  
count_FACTURACION_MES_SGTE         -0.217946       -0.002546  
count_PAGO_MENOS                    1.000000       -0.203399  
count_PAGO_MAS                     -0.203399        1.000000  
